<a href="https://colab.research.google.com/github/saad0O5/FlyRank-Internship-Work/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saad0O5/FlyRank-s-Project/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [29]:
import os, getpass

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

%pip -q install duckdb huggingface_hub
import duckdb, pandas as pd, numpy as np

con = duckdb.connect()
con.execute("PRAGMA threads=1")   # forces deterministic aggregation order
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"}
print("Connected.")

Paste your Hugging Face READ token (hf_...): ··········
Connected.


## 1. Method choice and why

**Method: Random Forest classifier**, scored as a ranking (probability output), same
shape as the starter pipeline's best model (w01/w02: Precision@50 0.240 → 0.740 on
the starter data). Chosen over logistic regression because Week 2 already showed the
signals are tangled and partly non-linear (content_type/age/client interactions on
the starter data; here, CTR-vs-position and activity both matter but Signal B showed
activity's relationship isn't a simple monotonic one). A tree ensemble can combine
these without me hand-engineering interaction terms, and it stays interpretable via
feature importances, unlike a black-box alternative. Gradient boosting was considered
but random forest is used first as the same "next step up from the baseline" the
starter pipeline established, keeping the comparison consistent across notebooks.

In [30]:
feat = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp_prev90,
           SUM(gsc_clicks)      AS clk_prev90,
           AVG(gsc_avg_position) AS pos_avg_prev90,
           SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS ctr_prev90,
           COUNT(*) FILTER (WHERE gsc_impressions > 0) AS days_active_prev90
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2025-12-31' AND '2026-03-31'
    GROUP BY 1, 2
    HAVING imp_prev90 >= 100
    ORDER BY client_hash_id, content_hash_id
""").df()

label = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_next30
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-04-01' AND '2026-04-30'
    GROUP BY 1, 2
    ORDER BY client_hash_id, content_hash_id
""").df()

data = feat.merge(label, on=['client_hash_id', 'content_hash_id'], how='inner')
data['is_declining_future'] = (data['imp_next30'] < 0.8 * (data['imp_prev90'] / 3)).astype(int)
data['position_tier'] = pd.cut(data['pos_avg_prev90'], bins=[0, 3, 10, 20, 9999],
                                 labels=['1-3', '4-10', '11-20', '21+'])

print(f"{len(data):,} rows, decline rate: {data['is_declining_future'].mean():.3f}")
data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

115,617 rows, decline rate: 0.371


,client_hash_id,content_hash_id,imp_prev90,clk_prev90,pos_avg_prev90,ctr_prev90,days_active_prev90,imp_next30,is_declining_future,position_tier
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,865.0,6.0,15.197534,0.006936,91,561.0,0,11-20
1,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,318.0,0.0,9.713898,0.000000,47,0.0,1,4-10
2,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,380.0,0.0,9.036379,0.000000,89,23.0,1,4-10
3,client_0797ff3a1fc9a6a5,content_1207efddce873942,973.0,3.0,13.435947,0.003083,75,964.0,0,11-20
4,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,731.0,0.0,12.291710,0.000000,89,248.0,0,11-20


## 2. Split design

**Client-grouped split (GroupShuffleSplit on `client_hash_id`)**, not a plain random
split. Week 1's starter-data work already showed decline rate varies enormously by
client (0% to 93.7%), and w04's first baseline draft accidentally over-concentrated
its top-20 in a single client before that was caught and fixed. If pages from the
same client land in both train and test, the model could just memorize client-level
quirks rather than learn generalizable signal — a random split would let that happen
silently. A time-aware split isn't used here because the label is already a genuine
forward-looking window (prev-90 → next-30); the remaining risk is client leakage, not
time leakage, so client-grouping is the split that matches this specific risk.

In [31]:
from sklearn.model_selection import GroupShuffleSplit

feature_cols = ['imp_prev90', 'clk_prev90', 'pos_avg_prev90', 'ctr_prev90', 'days_active_prev90']
model_data = data.dropna(subset=feature_cols + ['is_declining_future'])
X, y, groups = model_data[feature_cols], model_data['is_declining_future'], model_data['client_hash_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])
print(f"Train: {len(X_tr):,} rows, {len(train_clients)} clients")
print(f"Test:  {len(X_te):,} rows, {len(test_clients)} clients")
print(f"Client overlap between train/test: {len(train_clients & test_clients)} (should be 0)")

Train: 105,653 rows, 33 clients
Test:  9,964 rows, 11 clients
Client overlap between train/test: 0 (should be 0)


## 3. Train + compare vs my baseline

Same data, same metric (Precision@50), same client-grouped test split as the model.
The baseline is my Week-4 rule (`ctr_gap`, tier-adjusted CTR-vs-position) — scored
on the identical test set so the comparison is fair.

In [32]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
model_proba = model.predict_proba(X_te)[:, 1]

def precision_at_k(y_true, scores, k=50):
    order = np.argsort(-scores)[:k]
    return y_true.iloc[order].mean()

model_p50 = precision_at_k(y_te, model_proba, k=50)
model_auc = roc_auc_score(y_te, model_proba)

# Baseline: recompute the w04 tier-adjusted ctr_gap score on the SAME test rows
test_baseline = model_data.loc[X_te.index].copy()
tier_avg_ctr = model_data.groupby('position_tier', observed=True)['ctr_prev90'].transform('mean')
test_baseline['ctr_gap'] = (tier_avg_ctr.loc[X_te.index] - test_baseline['ctr_prev90']).clip(lower=0)

baseline_p50 = precision_at_k(y_te, test_baseline['ctr_gap'].values, k=50)
baseline_auc = roc_auc_score(y_te, test_baseline['ctr_gap'])

print(f"{'Method':<20}{'Precision@50':>15}{'AUC':>10}")
print(f"{'Baseline (rule)':<20}{baseline_p50:>15.3f}{baseline_auc:>10.3f}")
print(f"{'Random Forest':<20}{model_p50:>15.3f}{model_auc:>10.3f}")
print(f"\nLift: {model_p50/baseline_p50:.1f}x on Precision@50" if baseline_p50 > 0 else "\nBaseline P@50 is 0 — lift undefined")

Method                 Precision@50       AUC
Baseline (rule)               0.340     0.528
Random Forest                 0.680     0.720

Lift: 2.0x on Precision@50


## 4. Errors and interpretation

**What the model leans on:** `pos_avg_prev90` (0.328) and `imp_prev90` (0.273)
dominate, together carrying ~60% of the importance — position and volume matter
most. `ctr_prev90` (0.174) and `days_active_prev90` (0.160) contribute meaningfully
but less. `clk_prev90` (0.065) matters least, likely because it's largely redundant
with `ctr_prev90` and `imp_prev90` combined (clicks ≈ impressions × CTR).

**A pattern in the errors, checked and mostly ruled out:** 4 of the 5 worst false
negatives and 3 of the 5 worst false positives involve `client_2094c6eb080311d5`.
Checking this client directly (Section 4 follow-up below) shows it's not actually
special — its true decline rate (23.6%) and average model score (26.8%) are close,
so it surfaces in "worst 5" lists mainly because it's one of the largest clients in
the test set (n=2,361), not because the model is uniquely wrong about it.

**The real calibration gap is elsewhere:** grouping all errors by client shows
`client_3f0ce4d44fe94f3d` (n=2,393) and `client_1a730cb2640a1abf` (n=1,321) are
overpredicted by roughly 2x (true decline rates 9.4% and 6.7% vs. average scores
19.1% and 17.2%), while `client_9958f0a7ae1df715` (n=2,426) is underpredicted
(84.7% true decline rate vs. only 38.5% average score). These three clients, not
the one that dominated the raw error sample, are where the model's calibration is
actually weakest.

**Extreme scores are rare, not a red flag:** only 220/9,964 rows (2.2%) score
exactly 0.0, and just 6/9,964 (0.06%) reach ≥0.9. The full score distribution
(mean 0.260, median 0.215, IQR 0.080–0.400) looks like an ordinary smooth output —
the extremes in the worst-5 sample are simply the tail of a normal distribution.

**Reproducibility note:** early runs of this notebook produced different numbers
between runs despite fixed random seeds (e.g. baseline P@50 ranging 0.340–0.380,
lift 1.6x–2.0x) — traced to DuckDB's default multi-threaded execution returning
Parquet rows in a non-deterministic order, which shifted floating-point aggregation
results and downstream tie-breaks. Fixed with `PRAGMA threads=1` plus explicit
`ORDER BY` on the feature/label queries; two consecutive full runs with this fix
produced identical results to 3+ decimal places, confirming stability. The numbers
below are from that verified-stable configuration.

**Overall:** a genuine 2.0x Precision@50 lift over the tier-adjusted baseline
(0.340 → 0.680) on a client-grouped test split is a real, defensible, and now
reproducible result — smaller than the starter dataset's 3.1x lift, which makes
sense given this is a true forward-looking label on a much larger, noisier real
dataset rather than a current-state proxy on a clean 30K-row sample.

**Revised takeaway:** the model's real weakness isn't any single client dominating
errors — it's systematic per-client miscalibration in both directions (some
clients over-flagged, some under-flagged), which the aggregate Precision@50 (0.680)
doesn't reveal on its own. A client-level calibration step, or client-level base
rate as a feature (with leakage caution, since a client's true future decline rate
isn't knowable in advance), would be a natural next step for Week 6 to investigate.

In [33]:
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Feature importances:")
print(importances.round(3))

# Look at a handful of the model's biggest misses
test_results = X_te.copy()
test_results['y_true'] = y_te.values
test_results['model_score'] = model_proba
test_results['client_hash_id'] = groups.iloc[test_idx].values

false_negatives = test_results[(test_results['y_true'] == 1)].sort_values('model_score').head(5)
false_positives = test_results[(test_results['y_true'] == 0)].sort_values('model_score', ascending=False).head(5)

print("\nWorst false negatives (declined but model scored them low):")
print(false_negatives[['client_hash_id', 'imp_prev90', 'ctr_prev90', 'days_active_prev90', 'model_score']])

print("\nWorst false positives (model scored high but didn't decline):")
print(false_positives[['client_hash_id', 'imp_prev90', 'ctr_prev90', 'days_active_prev90', 'model_score']])

Feature importances:
pos_avg_prev90        0.328
imp_prev90            0.273
ctr_prev90            0.174
days_active_prev90    0.160
clk_prev90            0.065
dtype: float64

Worst false negatives (declined but model scored them low):
                client_hash_id  imp_prev90  ctr_prev90  days_active_prev90  \
10265  client_0fa64a184f18a4a0      3141.0    0.005412                  35   
18149  client_2094c6eb080311d5       269.0    0.000000                  22   
17522  client_2094c6eb080311d5       222.0    0.000000                  16   
36841  client_3f0ce4d44fe94f3d      1243.0    0.009654                  20   
19701  client_2094c6eb080311d5       838.0    0.004773                  16   

       model_score  
10265        0.000  
18149        0.000  
17522        0.000  
36841        0.005  
19701        0.005  

Worst false positives (model scored high but didn't decline):
                client_hash_id  imp_prev90  ctr_prev90  days_active_prev90  \
19085  client_2094c6eb08031

In [34]:
client_errors = test_results.groupby('client_hash_id').apply(
    lambda g: pd.Series({
        'n': len(g),
        'decline_rate': g['y_true'].mean(),
        'avg_model_score': g['model_score'].mean()
    })
).round(3)
print(client_errors.sort_values('n', ascending=False))

                              n  decline_rate  avg_model_score
client_hash_id                                                
client_9958f0a7ae1df715  2426.0         0.847            0.385
client_3f0ce4d44fe94f3d  2393.0         0.094            0.191
client_2094c6eb080311d5  2361.0         0.236            0.268
client_1a730cb2640a1abf  1321.0         0.067            0.172
client_0fa64a184f18a4a0   820.0         0.176            0.126
client_f623b01661d4bfe4   277.0         0.657            0.354
client_cd12bcfd98942aa1   267.0         0.318            0.358
client_def0955f7a377868    69.0         0.232            0.455
client_8dbf3abdf07569e0    19.0         0.684            0.319
client_0e1acc6cd57b0eba     7.0         0.286            0.335
client_e00b29e582949543     4.0         0.000            0.041


/tmp/ipykernel_1916/567694413.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  client_errors = test_results.groupby('client_hash_id').apply(


In [35]:
print(pd.Series(model_proba).describe())
print(f"\nRows with score exactly 0.0: {(model_proba == 0).sum()} / {len(model_proba)}")
print(f"Rows with score >= 0.9: {(model_proba >= 0.9).sum()} / {len(model_proba)}")

count    9964.000000
mean        0.259530
std         0.207691
min         0.000000
25%         0.080000
50%         0.215000
75%         0.400000
max         0.970000
dtype: float64

Rows with score exactly 0.0: 220 / 9964
Rows with score >= 0.9: 6 / 9964


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.